**Without parallelization**

In [10]:
import time

def work(x):
    time.sleep(2)
    return x


if __name__ == '__main__':
    numbers = [1, 2, 3, 4]

    result = []

    start_time = time.time()
    for n in numbers:
        result.append(work(n))

    print(time.time()-start_time)

8.000659227371216


**With Parallelization**

In [11]:
from multiprocessing import Pool
import time

def work(x):
    time.sleep(2)
    return x

if __name__ == "__main__":

    numbers = [1, 2, 3, 4]

    start_time = time.time()
    with Pool() as pool:
        result = pool.map(work, numbers)

    print(time.time()-start_time)

4.041322946548462


**Other Stuffs May Needed To develop Parallel Algorithms**

In [12]:
from multiprocessing import Lock, Array
import ctypes
from itertools import accumulate

if __name__ == "__main__":

    # Shared array
    arr = Array(ctypes.c_int, [10,20,30])

    # Lock
    lock = Lock()

    with lock:
        arr[0] += 5

    print(list(arr))

    # Prefix sum
    degree = [2,4,3]

    prefix = [0]
    prefix.extend(accumulate(degree[:-1]))

    print(prefix)

[15, 20, 30]
[0, 2, 6]


##lab task 04

In [13]:
from multiprocessing import Pool, Array, Lock, cpu_count
from itertools import accumulate
from collections import deque
import ctypes
import time


##Convert an edge list to CSR representation


In [14]:
def build_csr(n, edge_list, undirected=True):
    adj = [[] for _ in range(n)]

    for u, v in edge_list:
        adj[u].append(v)

        if undirected:
            adj[v].append(u)

    offsets = [0]
    edges = []

    for neighbors in adj:
        edges.extend(neighbors)
        offsets.append(len(edges))

    return offsets, edges


In [15]:
_PARENT = None
_LOCK = None
_OFFSETS = None
_EDGES = None


def init_worker(parent, lock, offsets, edges):
    global _PARENT, _LOCK, _OFFSETS, _EDGES

    _PARENT = parent
    _LOCK = lock
    _OFFSETS = offsets
    _EDGES = edges


##Expand one frontier vertex


In [16]:
def expand_vertex(task):
    v, starting_index = task

    begin = _OFFSETS[v]
    end = _OFFSETS[v + 1]

    block = [-1] * (end - begin)

    for j, edge_position in enumerate(range(begin, end)):
        neighbor = _EDGES[edge_position]

        with _LOCK:
            if _PARENT[neighbor] == -1:
                _PARENT[neighbor] = v
                block[j] = neighbor

    return starting_index, block


##Parallel BFS


In [17]:
def parallel_bfs(n, offsets, edges, source, processes=None, verbose=True):
    if processes is None:
        processes = min(4, cpu_count())

    # Shared parent array. -1 means "not discovered yet".
    parent = Array(ctypes.c_int, [-1] * n, lock=False)
    parent[source] = source

    lock = Lock()

    frontier = [source]
    levels = []
    traversal_order = []

    level_number = 0

    with Pool(
        processes=processes,
        initializer=init_worker,
        initargs=(parent, lock, offsets, edges)
    ) as pool:

        while frontier:
            levels.append(frontier.copy())
            traversal_order.extend(frontier)

            # Degree of every vertex in current frontier
            degrees = [
                offsets[v + 1] - offsets[v]
                for v in frontier
            ]

            # Exclusive prefix sum
            prefix = [0]
            prefix.extend(accumulate(degrees[:-1]))

            # Total number of outgoing-edge positions
            total_positions = sum(degrees)
            frontier_next = [-1] * total_positions

            # One task for each current-frontier vertex
            tasks = list(zip(frontier, prefix))

            # Expand all current frontier vertices in parallel
            results = pool.map(expand_vertex, tasks)

            # Put each returned block in its prefix-sum-assigned region
            for start_index, block in results:
                frontier_next[
                    start_index:start_index + len(block)
                ] = block

            # Filter all -1 values
            new_frontier = [
                vertex
                for vertex in frontier_next
                if vertex != -1
            ]

            if verbose:
                print(f"Level {level_number}")
                print("Frontier       :", frontier)
                print("Degrees        :", degrees)
                print("Prefix sum     :", prefix)
                print("FrontierNext   :", frontier_next)
                print("New frontier   :", new_frontier)
                print()

            frontier = new_frontier
            level_number += 1

    return list(parent), traversal_order, levels


## Test the Parallel BFS

Example graph:

- 0 — 1
- 0 — 2
- 1 — 3
- 1 — 4
- 2 — 4
- 2 — 5
- 3 — 6
- 4 — 5
- 5 — 6

Source vertex = `0`


In [18]:
if __name__ == "__main__":
    n = 7

    edge_list = [
        (0, 1),
        (0, 2),
        (1, 3),
        (1, 4),
        (2, 4),
        (2, 5),
        (3, 6),
        (4, 5),
        (5, 6)
    ]

    offsets, edges = build_csr(n, edge_list, undirected=True)

    print("Offsets:", offsets)
    print("Edges  :", edges)
    print()

    parent, order, levels = parallel_bfs(
        n,
        offsets,
        edges,
        source=0,
        processes=3,
        verbose=True
    )

    print("BFS levels     :", levels)
    print("Traversal order:", order)
    print("Parent array   :", parent)


Offsets: [0, 2, 5, 8, 10, 13, 16, 18]
Edges  : [1, 2, 0, 3, 4, 0, 4, 5, 1, 6, 1, 2, 5, 2, 4, 6, 3, 5]

Level 0
Frontier       : [0]
Degrees        : [2]
Prefix sum     : [0]
FrontierNext   : [1, 2]
New frontier   : [1, 2]

Level 1
Frontier       : [1, 2]
Degrees        : [3, 3]
Prefix sum     : [0, 3]
FrontierNext   : [-1, 3, 4, -1, -1, 5]
New frontier   : [3, 4, 5]

Level 2
Frontier       : [3, 4, 5]
Degrees        : [2, 3, 3]
Prefix sum     : [0, 2, 5]
FrontierNext   : [-1, 6, -1, -1, -1, -1, -1, -1]
New frontier   : [6]

Level 3
Frontier       : [6]
Degrees        : [2]
Prefix sum     : [0]
FrontierNext   : [-1, -1]
New frontier   : []

BFS levels     : [[0], [1, 2], [3, 4, 5], [6]]
Traversal order: [0, 1, 2, 3, 4, 5, 6]
Parent array   : [0, 0, 0, 1, 1, 2, 3]
